# Related Works Style Analysis: 2015 vs 2025

**Hypothesis:** LLM availability (post-2022) has measurably altered the linguistic style, vocabulary, and structure of "Related Works" sections in Computer Science papers on arXiv.

**Date:** 2026-04-07

In [ ]:
import pathlib, re, collections, warnings, sys, ssl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind, chi2_contingency, fisher_exact
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag, pos_tag_sents

# Handle SSL certificate issues for NLTK downloads
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

DATA_2015 = pathlib.Path("data/txt")
DATA_2025 = pathlib.Path("data-2025/txt")
CIT_PATTERN = re.compile(r"<cit\.>")
REF_PATTERN = re.compile(r"<ref>")
GRAPHICS_PATTERN = re.compile(r"<\s*g\s*r\s*a\s*p\s*h\s*i\s*c\s*s\s*>")

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)

print("Setup complete.")

---
## Section 1: Data Loading

In [ ]:
def load_corpus(base_path, year_group):
    """Load all .txt files from a data directory into a list of dicts."""
    records = []
    base = pathlib.Path(base_path)
    assert base.exists(), f"Directory not found: {base}"

    for entry in sorted(base.iterdir()):
        if not entry.is_dir():
            continue
        # Check for nested subdirectories (e.g. cs/0006023)
        txt_files = list(entry.glob("*.txt"))
        subdirs = [d for d in entry.iterdir() if d.is_dir()]

        if txt_files:
            arxiv_id = entry.name
            for txt_file in sorted(txt_files):
                text = txt_file.read_text(encoding="utf-8", errors="replace")
                records.append({
                    "arxiv_id": arxiv_id,
                    "year_group": year_group,
                    "raw_text": text,
                    "filename": txt_file.name,
                })
        for sub in subdirs:
            arxiv_id = f"{entry.name}/{sub.name}"
            for txt_file in sorted(sub.glob("*.txt")):
                text = txt_file.read_text(encoding="utf-8", errors="replace")
                records.append({
                    "arxiv_id": arxiv_id,
                    "year_group": year_group,
                    "raw_text": text,
                    "filename": txt_file.name,
                })
    return records

records_2015 = load_corpus(DATA_2015, "2015")
records_2025 = load_corpus(DATA_2025, "2025")

df_files = pd.DataFrame(records_2015 + records_2025)
print(f"Loaded {len(df_files)} files from {df_files['arxiv_id'].nunique()} papers")
print(f"  2015 group: {df_files[df_files['year_group']=='2015']['arxiv_id'].nunique()} papers, {len(df_files[df_files['year_group']=='2015'])} files")
print(f"  2025 group: {df_files[df_files['year_group']=='2025']['arxiv_id'].nunique()} papers, {len(df_files[df_files['year_group']=='2025'])} files")

In [ ]:
# Corpus statistics
df_files["_wc"] = df_files["raw_text"].str.split().str.len()

corpus_stats = df_files.groupby("year_group").agg(
    n_papers=("arxiv_id", "nunique"),
    n_files=("raw_text", "count"),
    total_words=("_wc", "sum"),
    mean_words_per_file=("_wc", "mean"),
    median_words_per_file=("_wc", "median"),
).round(1)

display(corpus_stats)

In [ ]:
# Per-paper aggregation: concatenate all txt files per arxiv_id
df_papers = (
    df_files
    .sort_values(["arxiv_id", "filename"])
    .groupby(["arxiv_id", "year_group"], as_index=False)
    .agg(text=("raw_text", "\n\n".join), n_files=("filename", "count"))
)

# Clean markup for NLP processing
df_papers["text_clean"] = (
    df_papers["text"]
    .str.replace(GRAPHICS_PATTERN, " ", regex=True)
    .str.replace(REF_PATTERN, " ", regex=True)
    .str.replace(CIT_PATTERN, " CITATION ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Filter out papers with < 50 words
df_papers["_wc"] = df_papers["text_clean"].str.split().str.len()
n_before = len(df_papers)
df_papers = df_papers[df_papers["_wc"] >= 50].copy()
print(f"Papers before filter: {n_before}")
print(f"Papers after filter (>= 50 words): {len(df_papers)}")
print(f"  2015: {(df_papers['year_group']=='2015').sum()}")
print(f"  2025: {(df_papers['year_group']=='2025').sum()}")

### Data Notes

- All txt files are related work sections (pipeline ran with `txt_related_only: true`)
- Multiple files per paper represent subsections, concatenated above
- `<cit.>` placeholders replaced with `CITATION` token for NLP; original text used for citation counting
- `<ref>` and `< g r a p h i c s >` markup stripped
- Papers with fewer than 50 words filtered as noise/extraction failures

---
## Section 2: Text Preprocessing

In [ ]:
# --- Helper functions ---

def tokenize_sentences(text):
    """Sentence-tokenize text, filtering empty results."""
    return [s for s in sent_tokenize(text) if s.strip()]


def tokenize_words(text):
    """Word-tokenize text, keeping only alphanumeric tokens (no CITATION placeholder)."""
    return [t for t in word_tokenize(text) if t.isalpha() and t != "CITATION"]


def count_citations(text):
    """Count <cit.> occurrences in original text."""
    return len(CIT_PATTERN.findall(text))


def count_paragraphs(text):
    """Count paragraphs (split on blank lines)."""
    paras = [p for p in re.split(r"\n\s*\n", text.strip()) if p.strip()]
    return max(len(paras), 1)


def count_syllables(word):
    """Count syllables via vowel groups. Minimum 1."""
    return max(1, len(re.findall(r"[aeiouy]+", word.lower())))


def flesch_reading_ease(n_words, n_sentences, n_syllables):
    """Flesch reading ease score."""
    if n_sentences == 0 or n_words == 0:
        return np.nan
    return 206.835 - 1.015 * (n_words / n_sentences) - 84.6 * (n_syllables / n_words)


def gunning_fog(words, n_sentences):
    """Gunning Fog index. Complex words = 3+ syllables."""
    n_words = len(words)
    if n_sentences == 0 or n_words == 0:
        return np.nan
    complex_count = sum(1 for w in words if count_syllables(w) >= 3)
    return 0.4 * ((n_words / n_sentences) + 100.0 * (complex_count / n_words))


_BE_FORMS = {"am", "is", "are", "was", "were", "be", "been", "being"}

def is_passive_tagged(tagged_tokens):
    """Check if a POS-tagged sentence contains passive voice (be-form + VBN within 3 tokens)."""
    for i, (word, tag) in enumerate(tagged_tokens):
        if word.lower() in _BE_FORMS:
            for j in range(i + 1, min(i + 4, len(tagged_tokens))):
                if tagged_tokens[j][1] == "VBN":
                    return True
    return False


def type_token_ratio(words):
    """Standard type-token ratio."""
    if not words:
        return np.nan
    return len(set(words)) / len(words)


def mattr(words, window=50):
    """Moving Average Type-Token Ratio (length-independent)."""
    if not words:
        return np.nan
    if len(words) < window:
        return type_token_ratio(words)
    ttrs = []
    for i in range(len(words) - window + 1):
        chunk = words[i : i + window]
        ttrs.append(len(set(chunk)) / window)
    return np.mean(ttrs)


def hapax_legomena_ratio(words):
    """Proportion of words appearing exactly once."""
    if not words:
        return np.nan
    freq = collections.Counter(words)
    hapax = sum(1 for c in freq.values() if c == 1)
    return hapax / len(words)


print("Helper functions defined.")

In [ ]:
# Feature extraction
results = []
total = len(df_papers)

for idx, (i, row) in enumerate(df_papers.iterrows()):
    if idx % 500 == 0:
        print(f"Processing paper {idx}/{total}...")

    text_orig = row["text"]
    text = row["text_clean"]

    # Sentence tokenization
    sentences = tokenize_sentences(text)
    n_sents = len(sentences)

    # Per-sentence word tokenization (single pass)
    sent_word_lists = [tokenize_words(s) for s in sentences]
    sent_lengths = [len(wl) for wl in sent_word_lists]

    # Flat word list
    words = [w for wl in sent_word_lists for w in wl]
    words_lower = [w.lower() for w in words]
    n_words = len(words)

    # Syllable count
    n_syllables = sum(count_syllables(w) for w in words)

    # Passive voice detection (batch POS tagging)
    if n_sents > 0:
        tagged_sents = pos_tag_sents([word_tokenize(s) for s in sentences])
        passive_count = sum(1 for ts in tagged_sents if is_passive_tagged(ts))
    else:
        passive_count = 0

    # Citations (on original text)
    n_cit = count_citations(text_orig)

    results.append({
        "n_sentences": n_sents,
        "n_words": n_words,
        "n_paragraphs": count_paragraphs(text_orig),
        "n_citations": n_cit,
        "mean_sentence_length": np.mean(sent_lengths) if sent_lengths else np.nan,
        "median_sentence_length": np.median(sent_lengths) if sent_lengths else np.nan,
        "ttr": type_token_ratio(words_lower),
        "mattr": mattr(words_lower, window=50),
        "hapax_ratio": hapax_legomena_ratio(words_lower),
        "flesch": flesch_reading_ease(n_words, n_sents, n_syllables),
        "gunning_fog": gunning_fog(words, n_sents),
        "passive_ratio": passive_count / n_sents if n_sents > 0 else np.nan,
        "cit_per_sentence": n_cit / n_sents if n_sents > 0 else np.nan,
        "cit_per_100_words": n_cit / n_words * 100 if n_words > 0 else np.nan,
        "sentence_lengths": sent_lengths,
    })

df_features = pd.DataFrame(results, index=df_papers.index)
df_papers = pd.concat([df_papers, df_features], axis=1)

print(f"\nDone. Extracted features for {len(df_papers)} papers.")
df_papers[["arxiv_id", "year_group", "n_words", "n_sentences", "flesch", "mattr", "passive_ratio"]].head(10)

---
## Section 3: Linguistic Style Analysis

We compare six key linguistic metrics between the 2015 and 2025 groups:

| Metric | What it measures |
|--------|------------------|
| Mean sentence length | Syntactic complexity |
| MATTR (window=50) | Vocabulary richness (length-independent) |
| Hapax legomena ratio | Proportion of unique words (lexical diversity) |
| Flesch reading ease | Readability (higher = easier) |
| Gunning Fog index | Reading difficulty (higher = harder) |
| Passive voice ratio | Proportion of sentences with passive constructions |

In [ ]:
# Summary statistics table
LINGUISTIC_METRICS = ["mean_sentence_length", "mattr", "hapax_ratio", "flesch", "gunning_fog", "passive_ratio"]
METRIC_LABELS = {
    "mean_sentence_length": "Mean Sentence Length",
    "mattr": "MATTR (window=50)",
    "hapax_ratio": "Hapax Legomena Ratio",
    "flesch": "Flesch Reading Ease",
    "gunning_fog": "Gunning Fog Index",
    "passive_ratio": "Passive Voice Ratio",
    "n_words": "Section Word Count",
    "n_paragraphs": "Paragraph Count",
    "n_citations": "Citation Count",
    "cit_per_sentence": "Citations per Sentence",
    "cit_per_100_words": "Citations per 100 Words",
}

summary = df_papers.groupby("year_group")[LINGUISTIC_METRICS].agg(["mean", "median", "std", "min", "max"])
display(summary.T.style.format("{:.3f}"))

In [ ]:
# 2x3 violin plots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, metric in zip(axes.flat, LINGUISTIC_METRICS):
    sns.violinplot(data=df_papers, x="year_group", y=metric, ax=ax, cut=0, inner="quartile")
    ax.set_title(METRIC_LABELS.get(metric, metric))
    ax.set_xlabel("")
plt.suptitle("Linguistic Style Metrics: 2015 vs 2025", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Overlaid KDE of sentence length distributions (pooled across all sentences)
rows = []
for _, paper in df_papers.iterrows():
    for sl in paper["sentence_lengths"]:
        rows.append({"year_group": paper["year_group"], "sentence_length": sl})
df_sents = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 5))
for yg, color in [("2015", "C0"), ("2025", "C1")]:
    subset = df_sents[df_sents["year_group"] == yg]["sentence_length"]
    sns.kdeplot(subset, ax=ax, color=color, label=yg, clip=(0, 120), fill=True, alpha=0.3)
ax.set_xlabel("Sentence Length (words)")
ax.set_ylabel("Density")
ax.set_title("Sentence Length Distribution (all sentences pooled)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests for linguistic metrics
def compute_tests(df, metrics):
    """Run Mann-Whitney U and Welch's t-test for each metric, return results DataFrame."""
    results = []
    for metric in metrics:
        g2015 = df.loc[df["year_group"] == "2015", metric].dropna()
        g2025 = df.loc[df["year_group"] == "2025", metric].dropna()
        n1, n2 = len(g2015), len(g2025)

        # Mann-Whitney U
        u_stat, u_pval = mannwhitneyu(g2015, g2025, alternative="two-sided")
        rank_biserial = 1 - (2 * u_stat) / (n1 * n2)

        # Welch's t-test
        t_stat, t_pval = ttest_ind(g2015, g2025, equal_var=False)

        # Cohen's d
        pooled_std = np.sqrt(((n1 - 1) * g2015.std() ** 2 + (n2 - 1) * g2025.std() ** 2) / (n1 + n2 - 2))
        cohens_d = (g2015.mean() - g2025.mean()) / pooled_std if pooled_std > 0 else np.nan

        results.append({
            "Metric": METRIC_LABELS.get(metric, metric),
            "2015 Mean": g2015.mean(),
            "2025 Mean": g2025.mean(),
            "U Statistic": u_stat,
            "U p-value": u_pval,
            "Rank-Biserial r": rank_biserial,
            "t Statistic": t_stat,
            "t p-value": t_pval,
            "Cohen's d": cohens_d,
        })

    df_results = pd.DataFrame(results)
    return df_results

df_ling_tests = compute_tests(df_papers, LINGUISTIC_METRICS)
display(df_ling_tests.style.format({
    "2015 Mean": "{:.3f}", "2025 Mean": "{:.3f}",
    "U Statistic": "{:.0f}", "U p-value": "{:.2e}",
    "Rank-Biserial r": "{:.4f}", "t Statistic": "{:.2f}",
    "t p-value": "{:.2e}", "Cohen's d": "{:.4f}",
}))

### Interpretation

The statistical tests above reveal whether there are significant differences in linguistic style between the two eras. Key points to note:

- **Rank-biserial r** measures effect size: positive values mean the 2015 group scored higher; negative values mean the 2025 group scored higher.
- **Cohen's d** provides a standardized effect size: |d| < 0.2 is small, 0.2-0.5 is medium, > 0.8 is large.
- Given the large sample sizes (~4,000+ per group), even small effects may be statistically significant. Focus on effect sizes rather than p-values alone.

---
## Section 4: LLM-Specific Marker Analysis

Based on Liang et al. (2024) "Mapping the Increasing Use of LLMs in Scientific Papers", we examine two tiers of marker phrases:

1. **High-confidence markers**: Phrases that were rare in academic writing pre-LLM but became common in LLM outputs
2. **Elevated-frequency markers**: Phrases that existed before but spiked in usage post-LLM

In [ ]:
# Define marker phrase dictionaries
HIGH_CONFIDENCE_MARKERS = {
    "delve(s) into": re.compile(r"\bdelves?\s+into\b", re.I),
    "it is worth noting": re.compile(r"\bit\s+is\s+worth\s+noting\b", re.I),
    "comprehensive overview": re.compile(r"\bcomprehensive\s+overview\b", re.I),
    "in the realm of": re.compile(r"\bin\s+the\s+realm\s+of\b", re.I),
    "plays a crucial/pivotal role": re.compile(r"\bplays\s+a\s+(?:crucial|pivotal)\s+role\b", re.I),
    "noteworthy": re.compile(r"\bnoteworthy\b", re.I),
    "multifaceted": re.compile(r"\bmultifaceted\b", re.I),
    "tapestry": re.compile(r"\btapestry\b", re.I),
    "groundbreaking": re.compile(r"\bgroundbreaking\b", re.I),
    "paving the way": re.compile(r"\bpaving\s+the\s+way\b", re.I),
    "has garnered significant/considerable attention": re.compile(r"\bhas\s+garnered\s+(?:significant|considerable)\s+attention\b", re.I),
    "demonstrated remarkable": re.compile(r"\bdemonstrated\s+remarkable\b", re.I),
    "leveraging the power": re.compile(r"\bleveraging\s+the\s+power\b", re.I),
    "intricate": re.compile(r"\bintricate\b", re.I),
    "underscores": re.compile(r"\bunderscores\b", re.I),
}

ELEVATED_FREQ_MARKERS = {
    "furthermore": re.compile(r"\bfurthermore\b", re.I),
    "moreover": re.compile(r"\bmoreover\b", re.I),
    "additionally": re.compile(r"\badditionally\b", re.I),
    "landscape": re.compile(r"\blandscape\b", re.I),
    "notably": re.compile(r"\bnotably\b", re.I),
    "cutting-edge": re.compile(r"\bcutting[\s-]edge\b", re.I),
    "in recent years": re.compile(r"\bin\s+recent\s+years\b", re.I),
    "a growing body of": re.compile(r"\ba\s+growing\s+body\s+of\b", re.I),
    "shed(s) light on": re.compile(r"\bsheds?\s+light\s+on\b", re.I),
}

ALL_MARKERS = {**HIGH_CONFIDENCE_MARKERS, **ELEVATED_FREQ_MARKERS}
MARKER_TIER = {m: "high-confidence" for m in HIGH_CONFIDENCE_MARKERS}
MARKER_TIER.update({m: "elevated-frequency" for m in ELEVATED_FREQ_MARKERS})

print(f"Defined {len(HIGH_CONFIDENCE_MARKERS)} high-confidence + {len(ELEVATED_FREQ_MARKERS)} elevated-frequency = {len(ALL_MARKERS)} total markers")

In [ ]:
# Compute marker counts per paper
marker_raw = pd.DataFrame(index=df_papers.index)
marker_per1k = pd.DataFrame(index=df_papers.index)
marker_binary = pd.DataFrame(index=df_papers.index)

for label, pattern in ALL_MARKERS.items():
    counts = df_papers["text_clean"].apply(lambda t: len(pattern.findall(t)))
    marker_raw[label] = counts
    marker_per1k[label] = counts / df_papers["n_words"] * 1000
    marker_binary[label] = (counts > 0).astype(int)

print(f"Marker counts computed for {len(df_papers)} papers x {len(ALL_MARKERS)} markers")
print(f"\nTop markers by total occurrences (2025 group):")
mask_2025 = df_papers["year_group"].values == "2025"
display(marker_raw[mask_2025].sum().sort_values(ascending=False).head(10))

In [ ]:
# Grouped horizontal bar chart: % of papers containing each marker
pct_by_year = marker_binary.copy()
pct_by_year["year_group"] = df_papers["year_group"].values
pct = pct_by_year.groupby("year_group").mean() * 100

# Sort by ratio 2025/2015
ratio = pct.loc["2025"] / pct.loc["2015"].replace(0, 0.01)
order = ratio.sort_values(ascending=True).index

fig, ax = plt.subplots(figsize=(11, 9))
y_pos = np.arange(len(order))
bar_width = 0.35
ax.barh(y_pos - bar_width / 2, pct.loc["2015"][order], bar_width, label="2015", color="C0")
ax.barh(y_pos + bar_width / 2, pct.loc["2025"][order], bar_width, label="2025", color="C1")
ax.set_yticks(y_pos)
ax.set_yticklabels(order)
ax.set_xlabel("% of papers containing marker")
ax.set_title("LLM-Associated Marker Prevalence: 2015 vs 2025")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests per marker
def benjamini_hochberg(p_values):
    """Return Benjamini-Hochberg FDR-adjusted p-values."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    sorted_idx = np.argsort(p)
    sorted_p = p[sorted_idx]
    adjusted = np.empty(n)
    adjusted[sorted_idx[-1]] = min(sorted_p[-1], 1.0)
    for i in range(n - 2, -1, -1):
        adjusted[sorted_idx[i]] = min(adjusted[sorted_idx[i + 1]], sorted_p[i] * n / (i + 1))
    return np.clip(adjusted, 0, 1)


n_2015 = (df_papers["year_group"] == "2015").sum()
n_2025 = (df_papers["year_group"] == "2025").sum()
mask_15 = df_papers["year_group"].values == "2015"
mask_25 = df_papers["year_group"].values == "2025"

marker_test_results = []
for label in ALL_MARKERS:
    a = int(marker_binary.loc[mask_15, label].sum())  # 2015 contains
    b = int(n_2015 - a)
    c = int(marker_binary.loc[mask_25, label].sum())  # 2025 contains
    d = int(n_2025 - c)
    table = np.array([[a, b], [c, d]])

    # Choose test based on expected counts
    expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / table.sum()
    if (expected < 5).any():
        or_val, p_val = fisher_exact(table, alternative="two-sided")
        test_used = "Fisher"
    else:
        chi2, p_val, dof, _ = chi2_contingency(table)
        or_val = (a * d) / (b * c) if (b * c) > 0 else np.inf
        test_used = "Chi-squared"

    # Mann-Whitney on normalized frequencies
    freq_15 = marker_per1k.loc[mask_15, label]
    freq_25 = marker_per1k.loc[mask_25, label]
    u_stat, u_pval = mannwhitneyu(freq_15, freq_25, alternative="two-sided")

    marker_test_results.append({
        "Marker": label,
        "Tier": MARKER_TIER[label],
        "% 2015": a / n_2015 * 100,
        "% 2025": c / n_2025 * 100,
        "Odds Ratio": or_val,
        "Test": test_used,
        "p-value": p_val,
        "MW U p-value": u_pval,
    })

df_marker_tests = pd.DataFrame(marker_test_results)

# Multiple comparison corrections
raw_p = df_marker_tests["p-value"].values
df_marker_tests["Bonferroni p"] = np.minimum(raw_p * len(raw_p), 1.0)
df_marker_tests["BH FDR p"] = benjamini_hochberg(raw_p)
df_marker_tests["Significant (BH)"] = df_marker_tests["BH FDR p"] < 0.05

# Sort by odds ratio descending
df_marker_tests = df_marker_tests.sort_values("Odds Ratio", ascending=False)

display(df_marker_tests.style.format({
    "% 2015": "{:.1f}", "% 2025": "{:.1f}",
    "Odds Ratio": "{:.2f}", "p-value": "{:.2e}",
    "MW U p-value": "{:.2e}", "Bonferroni p": "{:.2e}", "BH FDR p": "{:.2e}",
}))

In [ ]:
# Aggregate LLM marker score: sum of high-confidence markers per 1000 words
hc_labels = list(HIGH_CONFIDENCE_MARKERS.keys())
df_papers["llm_marker_score"] = marker_per1k[hc_labels].sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df_papers, x="year_group", y="llm_marker_score", ax=ax, cut=0, inner="quartile")
ax.set_ylabel("High-Confidence LLM Markers per 1000 Words")
ax.set_xlabel("")
ax.set_title("Aggregate LLM Marker Score")
plt.tight_layout()
plt.show()

g15 = df_papers.loc[df_papers["year_group"] == "2015", "llm_marker_score"].dropna()
g25 = df_papers.loc[df_papers["year_group"] == "2025", "llm_marker_score"].dropna()
u_stat, u_pval = mannwhitneyu(g15, g25, alternative="two-sided")
r_rb = 1 - (2 * u_stat) / (len(g15) * len(g25))
print(f"2015 mean: {g15.mean():.4f}, 2025 mean: {g25.mean():.4f}")
print(f"Mann-Whitney U = {u_stat:.0f}, p = {u_pval:.2e}, rank-biserial r = {r_rb:.4f}")

---
## Section 5: Structural Pattern Analysis

This section examines changes in the structural characteristics of Related Works sections:
- Section length (word count)
- Number of paragraphs
- Citation count and density
- Citation introduction patterns

In [ ]:
# Summary statistics for structural metrics
STRUCTURAL_METRICS = ["n_words", "n_paragraphs", "n_citations", "cit_per_sentence", "cit_per_100_words"]
struct_summary = df_papers.groupby("year_group")[STRUCTURAL_METRICS].agg(["mean", "median", "std", "min", "max"])
display(struct_summary.T.style.format("{:.2f}"))

In [ ]:
# 2x2 box plots for structural metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
box_metrics = ["n_words", "n_paragraphs", "n_citations", "cit_per_sentence"]
for ax, metric in zip(axes.flat, box_metrics):
    sns.boxplot(data=df_papers, x="year_group", y=metric, ax=ax, showfliers=False)
    ax.set_title(METRIC_LABELS.get(metric, metric))
    ax.set_xlabel("")
plt.suptitle("Structural Metrics: 2015 vs 2025", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Citation introduction pattern analysis
VERB_PATTERN = re.compile(
    r"\b(?:proposed|introduced|shown|described|presented|discussed|studied|demonstrated|developed|designed|explored|investigated)"
    r"\s+(?:by|in)\s*$", re.I
)
AUTHOR_PATTERN = re.compile(r"(?:et\s+al\.|[A-Z][a-z]+(?:\s+and\s+[A-Z][a-z]+)?)\s*$")
LIST_PATTERN = re.compile(r"CITATION\s*[,;]\s*$")

def classify_citation_contexts(text_clean, text_orig):
    """Classify each <cit.> by its introduction pattern."""
    patterns = {"parenthetical": 0, "author_named": 0, "verb_led": 0, "list_style": 0}
    for match in CIT_PATTERN.finditer(text_orig):
        start = match.start()
        context = text_orig[max(0, start - 80):start].strip()
        # Replace <cit.> with CITATION in context for pattern matching
        context_clean = CIT_PATTERN.sub("CITATION", context)

        if LIST_PATTERN.search(context_clean):
            patterns["list_style"] += 1
        elif AUTHOR_PATTERN.search(context):
            patterns["author_named"] += 1
        elif VERB_PATTERN.search(context):
            patterns["verb_led"] += 1
        else:
            patterns["parenthetical"] += 1
    return patterns

# Apply to all papers
cit_patterns = df_papers.apply(
    lambda row: classify_citation_contexts(row["text_clean"], row["text"]), axis=1
).apply(pd.Series)

cit_patterns["year_group"] = df_papers["year_group"].values
cit_patterns["total"] = cit_patterns[["parenthetical", "author_named", "verb_led", "list_style"]].sum(axis=1)

# Data quality check
for yg in ["2015", "2025"]:
    n_total = (df_papers["year_group"] == yg).sum()
    n_zero = (df_papers.loc[df_papers["year_group"] == yg, "n_citations"] == 0).sum()
    print(f"{yg}: {n_zero}/{n_total} papers ({n_zero/n_total*100:.1f}%) have zero <cit.> markers")

# Proportions by year
cit_prop = cit_patterns.groupby("year_group")[["parenthetical", "author_named", "verb_led", "list_style"]].sum()
cit_prop_pct = cit_prop.div(cit_prop.sum(axis=1), axis=0) * 100
print("\nCitation introduction pattern distribution (%):")
display(cit_prop_pct.style.format("{:.1f}"))

In [ ]:
# Scatter plot: citation count vs word count
fig, ax = plt.subplots(figsize=(10, 7))
for yg, color in [("2015", "C0"), ("2025", "C1")]:
    subset = df_papers[df_papers["year_group"] == yg]
    ax.scatter(subset["n_words"], subset["n_citations"], alpha=0.1, s=8, c=color, label=yg)
    # Regression line (clip to 99th percentile)
    mask = subset["n_words"] <= subset["n_words"].quantile(0.99)
    z = np.polyfit(subset.loc[mask, "n_words"], subset.loc[mask, "n_citations"], 1)
    x_line = np.linspace(0, subset["n_words"].quantile(0.99), 100)
    ax.plot(x_line, np.polyval(z, x_line), color=color, linewidth=2)

ax.set_xlabel("Section Word Count")
ax.set_ylabel("Citation Count")
ax.set_title("Citation Count vs Section Length")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Statistical tests for structural metrics
df_struct_tests = compute_tests(df_papers, STRUCTURAL_METRICS)
display(df_struct_tests.style.format({
    "2015 Mean": "{:.2f}", "2025 Mean": "{:.2f}",
    "U Statistic": "{:.0f}", "U p-value": "{:.2e}",
    "Rank-Biserial r": "{:.4f}", "t Statistic": "{:.2f}",
    "t p-value": "{:.2e}", "Cohen's d": "{:.4f}",
}))

---
## Section 6: Combined Summary Dashboard

In [ ]:
# Master summary table
ALL_METRICS = LINGUISTIC_METRICS + STRUCTURAL_METRICS + ["llm_marker_score"]
METRIC_LABELS["llm_marker_score"] = "LLM Marker Score (HC per 1k words)"

master_results = []
for metric in ALL_METRICS:
    g15 = df_papers.loc[df_papers["year_group"] == "2015", metric].dropna()
    g25 = df_papers.loc[df_papers["year_group"] == "2025", metric].dropna()
    n1, n2 = len(g15), len(g25)

    u_stat, u_pval = mannwhitneyu(g15, g25, alternative="two-sided")
    r_rb = 1 - (2 * u_stat) / (n1 * n2)

    pooled_std = np.sqrt(((n1 - 1) * g15.std() ** 2 + (n2 - 1) * g25.std() ** 2) / (n1 + n2 - 2))
    d = (g15.mean() - g25.mean()) / pooled_std if pooled_std > 0 else np.nan

    pct_change = (g25.mean() - g15.mean()) / g15.mean() * 100 if g15.mean() != 0 else np.nan

    master_results.append({
        "Metric": METRIC_LABELS.get(metric, metric),
        "2015 Mean": g15.mean(),
        "2015 Median": g15.median(),
        "2025 Mean": g25.mean(),
        "2025 Median": g25.median(),
        "% Change": pct_change,
        "U Statistic": u_stat,
        "p-value": u_pval,
        "Rank-Biserial r": r_rb,
        "Cohen's d": d,
        "_abs_r": abs(r_rb),
    })

df_master = pd.DataFrame(master_results)

# BH correction across all metrics
df_master["BH FDR p"] = benjamini_hochberg(df_master["p-value"].values)
df_master["Significant"] = df_master["BH FDR p"] < 0.05
df_master = df_master.sort_values("_abs_r", ascending=False).drop(columns="_abs_r")

display(df_master.style.format({
    "2015 Mean": "{:.3f}", "2015 Median": "{:.3f}",
    "2025 Mean": "{:.3f}", "2025 Median": "{:.3f}",
    "% Change": "{:+.1f}%", "U Statistic": "{:.0f}",
    "p-value": "{:.2e}", "Rank-Biserial r": "{:.4f}",
    "Cohen's d": "{:.4f}", "BH FDR p": "{:.2e}",
}))

In [ ]:
# Effect size dashboard: horizontal bar chart of rank-biserial r
fig, ax = plt.subplots(figsize=(10, 8))

df_plot = df_master.sort_values("Rank-Biserial r")
colors = ["C1" if sig else "0.6" for sig in df_plot["Significant"]]

# Confidence intervals (normal approximation)
n1 = (df_papers["year_group"] == "2015").sum()
n2 = (df_papers["year_group"] == "2025").sum()
r_vals = df_plot["Rank-Biserial r"].values
se_r = np.sqrt((1 - r_vals ** 2) * (n1 + n2 + 1) / (3 * n1 * n2))
ci_err = 1.96 * se_r

y_pos = np.arange(len(df_plot))
ax.barh(y_pos, df_plot["Rank-Biserial r"], xerr=ci_err, color=colors, edgecolor="white", capsize=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(df_plot["Metric"])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Rank-Biserial r (positive = 2015 higher)")
ax.set_title("Effect Sizes Across All Metrics")

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="C1", label="Significant (BH FDR p < 0.05)"),
    Patch(facecolor="0.6", label="Not significant"),
], loc="lower right")

plt.tight_layout()
plt.show()

## Discussion

### Key Findings

The analysis above compares Related Works sections from ~4,600 papers (2015-era) and ~4,000 papers (2025-era) across linguistic style, LLM-associated markers, and structural patterns. The master summary table and effect size dashboard provide an at-a-glance view of which metrics show meaningful shifts.

### Limitations

1. **No per-category metadata**: The txt files don't encode which CS subcategory (cs.CL vs cs.CV etc.) each paper belongs to. Topic shifts between 2015 and 2025 are a potential confounder.
2. **Field growth**: The sheer growth in CS paper volume means the 2025 pool may include more junior or non-native English authors.
3. **`<cit.>` handling**: Differences between pipeline extraction versions could affect citation counts. Papers with zero `<cit.>` markers may have had citations stripped rather than replaced.
4. **Sample size effects**: With ~4,000+ papers per group, even trivially small effects achieve statistical significance. Effect sizes (rank-biserial r, Cohen's d) are more informative than p-values.
5. **Passive voice detection**: POS-tag heuristic is ~80-85% accurate, but the same bias applies to both groups.

### Future Work

- **Per-subcategory analysis**: Join `arxiv_id` back to metadata to compare within specific CS subcategories
- **Temporal trends within 2022-2025**: Track marker frequency month-by-month to find the inflection point
- **Comparison with non-CS fields**: Test whether the effect is specific to CS or appears across all arXiv disciplines

---
## Section 7: Appendix

In [ ]:
# Export metrics to CSV
export_cols = ["arxiv_id", "year_group", "n_files", "n_sentences", "n_words", "n_paragraphs",
               "n_citations", "mean_sentence_length", "median_sentence_length",
               "ttr", "mattr", "hapax_ratio", "flesch", "gunning_fog",
               "passive_ratio", "cit_per_sentence", "cit_per_100_words", "llm_marker_score"]
df_papers[export_cols].to_csv("related_works_metrics.csv", index=False)
print(f"Exported {len(df_papers)} rows to related_works_metrics.csv")

In [ ]:
# Environment info for reproducibility
import platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print()
for pkg_name in ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "nltk"]:
    pkg = __import__(pkg_name)
    print(f"{pkg_name}: {pkg.__version__}")
print()
print(f"2015 data: {DATA_2015.resolve()}")
print(f"2025 data: {DATA_2025.resolve()}")